# Chapter 11: the panel as data
400 synthetic tasting records (50 sessions x 8 tasters from a pool of 18, seed 1102) joined to 50 bottle lots with TPO (seed 2081). The notebook rebuilds sections 11.1 to 11.5, maps raw comments to the controlled list and checks a drafted note against a fact sheet.

**No code tier.** Copy 20 rows of `ch11_tasting_records.csv` into a sheet. Count papery tags per lot, add each lot's TPO from `ch11_lots.csv` and use CORREL.

Synthetic data from Kestrel Brewing, an invented plant. Copyright 2026 Ankur Napa. All rights reserved. You may use these files for your own learning; do not republish or redistribute them.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = "../data/"
from scipy import stats

In [ ]:
r = pd.read_csv(DATA + "ch11_tasting_records.csv"); lots = pd.read_csv(DATA + "ch11_lots.csv"); spikes = pd.read_csv(DATA + "ch11_spikes.csv")
print(r.off_flavour.value_counts())
print("spikes named:", spikes.spikes_named_of_6.value_counts().sort_index(ascending=False).to_dict(), "| retrain:", (spikes.spikes_named_of_6 < 4).sum())

In [ ]:
tot = r.groupby("lot_id").bitterness.transform("sum")
r["offset"] = r.bitterness - (tot - r.bitterness) / 7
g = r.groupby("taster").offset
cal = pd.DataFrame({"records": g.size(), "mean offset": g.mean().round(2), "standard errors": (g.mean().abs() / (g.std() / np.sqrt(g.size()))).round(1)})
cal = cal.join(spikes.set_index("taster"))
cal[cal["mean offset"].abs() > 0.75]

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(cal.spikes_named_of_6 + np.random.default_rng(0).uniform(-0.15, 0.15, 18), cal["mean offset"].abs(), color="#0b4f45")
for t in ("T03", "T13"): plt.annotate(t, (cal.at[t, "spikes_named_of_6"], abs(cal.at[t, "mean offset"])))
plt.axvline(3.5, ls="--", color="#6b6b6b"); plt.axhline(0.75, ls="--", color="#6b6b6b")
plt.xlabel("spikes named of 6"); plt.ylabel("absolute offset from the room (points)"); plt.show()
safe = ((cal.spikes_named_of_6 >= 4) & (cal["mean offset"].abs() <= 0.75)).sum(); print(safe, "of 18 in the safe corner")

In [ ]:
band = {"malt": 4, "hop_aroma": 3, "bitterness": 4, "esters": 2, "body": 4, "finish": 4}
view = r.groupby("lot_id")[list(band)].mean()
view["not TTT votes"] = r.groupby("lot_id").true_to_type.apply(lambda v: (v == "no").sum())
view["papery"] = r.groupby("lot_id").off_flavour.apply(lambda v: (v == "papery").sum())
view["outside band"] = [", ".join(a for a in band if abs(view.at[l, a] - band[a]) > 1) for l in view.index]
view = view.join(lots.set_index("lot_id").tpo_ppb)
print((view["not TTT votes"] >= 4).sum(), "lots fail the half rule")
view[(view["not TTT votes"] >= 4) | (view["outside band"] != "")].round(2)

In [ ]:
share = view.papery / 8; tpo = view.tpo_ppb
rr = np.corrcoef(tpo, share)[0, 1]; slope = np.polyfit(tpo, share, 1)[0]
print(f"r {rr:.2f}  r squared {rr ** 2:.2f}  slope {1000 * slope:.1f} points per 10 ppb  t {rr * math.sqrt(48) / math.sqrt(1 - rr ** 2):.1f}")
print(f"above 120 ppb: {(tpo > 120).sum()} lots, mean papery {share[tpo > 120].mean():.1%}; the rest {share[tpo <= 120].mean():.1%}")
c = lots[lots.staling_complaint == 1]
print(len(c), "complaints;", (c.tpo_ppb > 140).sum(), "on the", (lots.tpo_ppb > 140).sum(), "lots above 140 ppb; chance of 4 or more:", round(stats.binom.sf(3, 6, 0.14), 4))
plt.scatter(tpo, share * 100, color="#2f7f72"); plt.scatter(c.tpo_ppb, view.loc[c.lot_id].papery / 8 * 100, facecolors="none", edgecolors="#b23a2e", s=90)
for v, col in ((100, "#2f7f72"), (120, "#c27c0e"), (150, "#b23a2e")): plt.axvline(v, ls="--", color=col)
plt.xlabel("TPO (ppb)"); plt.ylabel("papery share (percent)"); plt.show()

## Mapping raw words to the controlled list (11.4.1)

In [ ]:
mapping = pd.read_csv(DATA + "ch11_descriptor_map.csv")
lookup = dict(zip(mapping.word, mapping.controlled_term))
raw = pd.read_csv(DATA + "ch11_raw_comments.csv").comment
def map_comment(text):
    hits = sorted({term for word, term in lookup.items() if word in text.lower()})
    return ", ".join(hits) if hits else "queue for the panel leader"
pd.DataFrame({"comment": raw, "controlled term": raw.map(map_comment)})

## A fact sheet check on a drafted note (11.4.2)
The draft may only say what a field in the record says. Every sentence is checked against the fact sheet; a sentence with no field goes back to a person.

In [ ]:
lot = view[(view.tpo_ppb == 138) & (view.papery == 5)].index[0]
facts = {"papery": f"{int(view.at[lot, 'papery'])} of 8", "bitterness": f"{view.at[lot, 'bitterness']:.1f}", "tpo": f"{int(view.at[lot, 'tpo_ppb'])} ppb"}
draft = ["Five of eight tasters noted a papery character.", "Bitterness 4.1.", "TPO 138 ppb.", "A clean, crisp finish."]
words = {"five of eight": facts["papery"] == "5 of 8", "bitterness": facts["bitterness"] in " ".join(draft), "tpo": facts["tpo"] in " ".join(draft)}
for sentence in draft:
    s_ = sentence.lower()
    field = next((k for k in words if k in s_), None)
    print(("supported by the record" if field and words[field] else "NO FIELD: back to a person"), "|", sentence)

**What to look at.** T03 sits 1.68 points high and T12 never tags papery. Every lot is under the 150 ppb limit, yet the papery share climbs with TPO. The complaint cluster is planted, so learn the method from it, not a finding.